<a href="https://colab.research.google.com/github/chihhui5/Hands_On_Training/blob/main/W4_FineTunning_TinyLLaMa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1. install

In [ ]:
#transformers(main library for loading& fine-tuning mode(BERT, GPT, LLaMA, etc))
#datasets(for loading, preprocessing, and managing datasets (JSON, CSV, etc))
#accelerate(Helps you run training efficiently on GPUs/TPUs and handle mixed precision.)
#bitsandbytes(Allows 4-bit / 8-bit quantization — crucial for running large models on small GPUs.)
#peft(Parameter-Efficient Fine-Tuning)(fine-tune big models efficiently using LoRA, QLoRA, Prefix Tuning, etc.)
#trl(Transformer Reinforcement Learning)Provides tools for supervised fine-tuning (SFT) and RLHF (Reinforcement Learning from Human Feedback).
#sentencepiece(Tokenizer library used by models like LLaMA, T5, and others)

!pip install sentencepiece
!pip install transformers datasets peft accelerate bitsandbytes

Step 2. import kits

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import torch

Step 3. import model and tokenizer

In [8]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step 4.Establish instruction dataset

In [11]:
data = [
    {"instruction": "What is NVIDIA famous for?", "response": "NVIDIA is known for GPUs and AI computing."},
    {"instruction": "Explain what CUDA is.", "response": "CUDA is a parallel computing platform developed by NVIDIA."},
    {"instruction": "What is deep learning?", "response": "Deep learning is a subset of machine learning using neural networks."},
    {"instruction": "What is a GPU?", "response": "A GPU is a processor designed for parallel computations."}
]

dataset = Dataset.from_list(data)

Step 5. LLM training format

In [12]:
def format_prompt(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"

dataset = dataset.map(lambda x: {"text": format_prompt(x)})

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Step 6. Tokenization

In [17]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Step 7. LoRA Setting

In [14]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Step 8. Training Setting

In [15]:
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no"
)

Step 9.Training

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

tokenized_dataset.set_format(type="torch")
trainer.train()

Step,Training Loss
1,16.547602
2,16.547600
3,15.978095
4,15.444681
5,15.023169


TrainOutput(global_step=5, training_loss=15.90822925567627, metrics={'train_runtime': 3.5966, 'train_samples_per_second': 5.561, 'train_steps_per_second': 1.39, 'total_flos': 31814823444480.0, 'train_loss': 15.90822925567627, 'epoch': 5.0})

Step 10. Testing

In [20]:
def generate_response(prompt):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "### Instruction:\nWhat is CUDA?\n\n### Response:\n"
print(generate_response(prompt))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is CUDA?

### Response:
CUDA is a programming model for accelerating parallel computing on graphics processing units (GPUs). It provides a high-level programming interface for writing applications that use GPUs, allowing developers to write code that is optimized for the GPU and can run on multiple GPUs simultaneously. CUDA is designed to be a general-purpose programming model that can be used to accelerate a wide range of applications, including scientific computing, graphics rendering, and machine learning.


Step 11.Model Storage

In [21]:
model.save_pretrained("./tinyllama-lora")
tokenizer.save_pretrained("./tinyllama-lora")

('./tinyllama-lora/tokenizer_config.json',
 './tinyllama-lora/chat_template.jinja',
 './tinyllama-lora/tokenizer.json')